In [20]:
import pandas as pd
import numpy as np

# --- Configuration ---
DELIMITER = ';'
COLUMN_NAMES = ['seed', 'structure', 'x', 'z', 'details']
REPORT_SIZE = 7  # Change this to 5, 10, or any other number

# Structures to exclude (Nether and End dimensions)
EXCLUDE_STRUCTURES = [
    'fortress', 'bastion_remnant', 'end_city', 'end_gateway', 'ruined_portal_nether'
]

# --- Scoring Logic (Parity Interest Score) ---

# Structures most likely to be the same (High Parity: 3.0)
HIGH_PARITY_STRUCTURES = [
    'desert_pyramid',    # Very consistent
    'jungle_pyramid',    # Very consistent
    'swamp_hut',         # Very consistent
    'igloo',             # Very consistent
    'ocean_monument',    # Highly consistent coordinates
    'desert_well'        # Consistent (though rare)
]

# Other structures (Medium Parity/Neutral: 2.0)
MEDIUM_PARITY_STRUCTURES = [
    'mineshaft',         # Similar regions but sprawling/variable
    'ocean_ruin',        # Generally close but can shift
    'shipwreck',         # Similar areas but exact placement varies
    'buried_treasure',   # Close but not perfect
    'ruined_portal',     # Similar frequency but coordinates vary more
    'pillager_outpost',  # Decent but not perfect coordinate match
    'amethyst_geode',    # Similar regions
    'ancient_city',      # Generally consistent
    'trial_chambers',    # Newer, generally good
    'trail_ruins'        # Decent consistency
]

DEFAULT_WEIGHT = 1.0

In [21]:
def get_parity_weight(structure):
    """Assigns a weight based on cross-edition generation parity."""
    if structure in HIGH_PARITY_STRUCTURES:
        return 3.0
    elif structure in MEDIUM_PARITY_STRUCTURES:
        return 2.0
    else:
        return DEFAULT_WEIGHT

In [29]:
def calculate_report(df, report_size, distance_penalty=0.0005, max_structures_shown=8):
    """
    Calculates the Parity Interest Score and generates the final report.
    
    Args:
        df: DataFrame containing seed structure data with columns: seed, structure, x, z
        report_size: Number of top seeds to include in the report
        distance_penalty: Penalty factor for exponential distance decay (default: 0.0005)
        max_structures_shown: Maximum number of structure types to show in report (default: 8)
        
    Returns:
        DataFrame with top seeds ranked by Parity Interest Score (0-100 scale)
        
    Raises:
        ValueError: If df is empty or report_size is invalid
    """
    
    # Input validation
    if df.empty:
        raise ValueError("Input dataframe is empty")
    if report_size <= 0:
        raise ValueError("report_size must be positive")
    
    # 1. Filter and Calculate Distance
    df_overworld = df[~df['structure'].isin(EXCLUDE_STRUCTURES)].copy()
    
    if df_overworld.empty:
        raise ValueError("No structures remaining after filtering")
    
    df_overworld['distance'] = df_overworld['x'].abs() + df_overworld['z'].abs()
    
    # 2. Apply Weights and Score (vectorized)
    import numpy as np
    
    weight_map = {struct: get_parity_weight(struct) for struct in df_overworld['structure'].unique()}
    df_overworld['weight'] = df_overworld['structure'].map(weight_map)
    
    # Exponential Distance Decay Formula
    df_overworld['weighted_score'] = df_overworld['weight'] * np.exp(-distance_penalty * df_overworld['distance'])
    
    # 3. Aggregate Scores
    interest_scores = (df_overworld.groupby('seed')['weighted_score']
                      .sum()
                      .sort_values(ascending=False)
                      .reset_index()
                      .rename(columns={'weighted_score': 'Parity Interest Score'}))
    
    # Normalize scores to 0-100 scale
    min_score = interest_scores['Parity Interest Score'].min()
    max_score = interest_scores['Parity Interest Score'].max()
    
    if max_score > min_score:  # Avoid division by zero
        interest_scores['Parity Interest Score'] = (
            (interest_scores['Parity Interest Score'] - min_score) / 
            (max_score - min_score) * 100
        )
    else:
        interest_scores['Parity Interest Score'] = 100.0  # All seeds are equal
    
    # 4. Generate Report
    top_seeds = interest_scores.head(report_size)
    top_seed_list = top_seeds['seed'].tolist()
    
    # Filter data for top seeds only
    top_seed_data = df_overworld[df_overworld['seed'].isin(top_seed_list)]
    
    # Get all structure counts
    structure_counts = (top_seed_data.groupby('seed')['structure']
                       .value_counts()
                       .unstack(fill_value=0))
    
    # Dynamically select structures to show based on parity and presence
    high_parity_in_data = [s for s in HIGH_PARITY_STRUCTURES if s in structure_counts.columns]
    medium_parity_in_data = [s for s in MEDIUM_PARITY_STRUCTURES if s in structure_counts.columns]
    low_parity_in_data = [s for s in structure_counts.columns 
                          if s not in HIGH_PARITY_STRUCTURES and s not in MEDIUM_PARITY_STRUCTURES]
    
    # Prioritize high parity, then medium, then low, up to max_structures_shown
    report_structures = []
    
    # Add high parity first
    for struct in high_parity_in_data:
        if len(report_structures) < max_structures_shown:
            report_structures.append(struct)
    
    # Add medium parity next
    for struct in medium_parity_in_data:
        if len(report_structures) < max_structures_shown:
            report_structures.append(struct)
    
    # Fill remaining slots with low parity by frequency
    if len(report_structures) < max_structures_shown:
        low_parity_sorted = structure_counts[low_parity_in_data].sum().sort_values(ascending=False).index.tolist()
        for struct in low_parity_sorted:
            if len(report_structures) < max_structures_shown:
                report_structures.append(struct)
    
    # Get total structure count and average distance in one pass
    seed_stats = top_seed_data.groupby('seed').agg({
        'structure': 'size',
        'distance': 'mean'
    }).rename(columns={
        'structure': 'Total Structures (Overworld)',
        'distance': 'Avg Distance'
    })
    
    # Merge everything together
    # Only include structures that actually exist in structure_counts
    available_structures = [s for s in report_structures if s in structure_counts.columns]
    
    final_report = (top_seeds
                   .merge(structure_counts[available_structures].fillna(0).astype(int), 
                         on='seed', how='left')
                   .merge(seed_stats, on='seed', how='left'))
    
    # Fill NaN values for structures with 0 (in case merge created them)
    for struct in available_structures:
        if struct in final_report.columns:
            final_report[struct] = final_report[struct].fillna(0).astype(int)
    
    # 5. Format and Clean Up
    # Create parity labels for column names
    column_renames = {}
    for struct in report_structures:
        if struct in HIGH_PARITY_STRUCTURES:
            parity_label = "High Parity"
        elif struct in MEDIUM_PARITY_STRUCTURES:
            parity_label = "Medium Parity"
        else:
            parity_label = "Low Parity"
        
        # Convert structure name to title case and add parity label
        display_name = struct.replace('_', ' ').title()
        column_renames[struct] = f"{display_name} ({parity_label})"
    
    final_report = final_report.rename(columns=column_renames)
    
    # Round numeric columns
    final_report['Parity Interest Score'] = final_report['Parity Interest Score'].round(2)
    final_report['Avg Distance'] = final_report['Avg Distance'].round(0).astype(int)
    
    # Reorder columns: seed, then structure columns (in order), then stats
    structure_columns = [column_renames[s] for s in report_structures]
    column_order = ['seed'] + structure_columns + ['Total Structures (Overworld)', 'Avg Distance', 'Parity Interest Score']
    
    final_report = final_report[column_order]
    
    return final_report

In [28]:
# Load the data with corrected skiprows and names
FILE_NAME = "../data/analysis-inputs/structures-parity-input.csv"
df = pd.read_csv(FILE_NAME, sep=DELIMITER, skiprows=1, names=COLUMN_NAMES)

print(df.head())
print(df.info())

                  seed       structure     x     z details
0  3109735542699327489  desert_pyramid -1520 -2208     NaN
1  3109735542699327489  desert_pyramid -1216 -1520     NaN
2  3109735542699327489  desert_pyramid -1392  -896     NaN
3  3109735542699327489  desert_pyramid -1024  -768     NaN
4  3109735542699327489  desert_pyramid  -432 -1760     NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532617 entries, 0 to 532616
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   seed       532617 non-null  int64 
 1   structure  532617 non-null  object
 2   x          532617 non-null  int64 
 3   z          532617 non-null  int64 
 4   details    458520 non-null  object
dtypes: int64(3), object(2)
memory usage: 20.3+ MB
None


In [30]:
# Run the calculation
report_df = calculate_report(df, REPORT_SIZE)

# Display the report
print(f"Top {REPORT_SIZE} Seeds Report (Parity Interest Score):")
display(report_df)

Top 7 Seeds Report (Parity Interest Score):


,seed,Desert Pyramid (High Parity),Jungle Pyramid (High Parity),Swamp Hut (High Parity),Igloo (High Parity),Desert Well (High Parity),Mineshaft (Medium Parity),Ocean Ruin (Medium Parity),Shipwreck (Medium Parity),Total Structures (Overworld),Avg Distance,Parity Interest Score
0,-530017381146162810,4,3,0,0,10,345,130,96,4685,2391,100.00
1,6771162039751544850,1,2,3,0,0,354,107,82,4679,2398,97.32
2,-7827537627346628587,0,5,0,0,2,327,101,80,4663,2395,89.45
3,-3671278121237083460,0,0,3,0,0,400,87,63,4632,2396,88.62
4,-5386868104288534490,6,1,0,1,7,387,90,67,4660,2403,88.05
5,-1210905349809241909,2,2,2,6,1,376,96,66,4679,2413,86.72
6,1591459518322051252,0,0,0,4,0,356,113,88,4649,2401,86.65
